# T01 — Backbone Loader Check

Checks `anyup/backbones.py` (task T01, see `docs/plan/P1_experiments/T01_backbone_loader.md`).

1. Shapes, `meta`, guide alignment and guide normalization for every adapter
2. VideoMAE vs the Exp 2 extraction code (same DAVIS frames)
3. Swin-T per-frame vs the Exp 3 extraction code (same DAVIS frames)
4. DINOv2 (timm wrapper): pretrained weights are loaded, features are sensible

Run all cells on a GPU runtime, then paste the **final summary cell's output** back.

In [ ]:
# ── Cell 1 · Drive + repo ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess
REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "3Dconv"
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
subprocess.run(["pip", "install", "-e", REPO_DIR, "-q"], check=True)
subprocess.run(["pip", "install", "transformers", "timm", "-q"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# ── Cell 2 · Imports + config ───────────────────────────────────────────────
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image

from anyup.backbones import (BackboneLoader, IMAGENET_MEAN, IMAGENET_STD,
                             resize_frames, normalize_frames)

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
DAVIS_ZIP  = "/content/drive/MyDrive/results/DAVIS-2017-trainval-480p.zip"
DAVIS_ROOT = "/content/DAVIS"
IMG_SIZE   = 224   # backbone input size used by Exp 2 / Exp 3; all adapters here use the same

# Tolerance for "exact" regression checks (same input, same weights; only CPU vs GPU
# normalization arithmetic and cuDNN nondeterminism may differ)
EXACT_MAX_ABS = 1e-3
EXACT_MIN_COS = 0.99999

RESULTS = []   # (check name, passed: bool | None for informational, detail str)
def record(name, passed, detail):
    RESULTS.append((name, passed, detail))
    tag = "INFO" if passed is None else ("PASS" if passed else "FAIL")
    print(f"[{tag}] {name}: {detail}")

def compare(a, b):
    a, b = a.float().cpu(), b.float().cpu()
    max_abs = (a - b).abs().max().item()
    cos = F.cosine_similarity(a.flatten(1), b.flatten(1), dim=1).min().item()   # worst frame
    return max_abs, cos

print("Device:", DEVICE, "| backbones:", BackboneLoader.available())

In [ ]:
# ── Cell 3 · DAVIS frames ───────────────────────────────────────────────────
if not os.path.exists(os.path.join(DAVIS_ROOT, "ImageSets")):
    subprocess.run(["unzip", "-q", DAVIS_ZIP, "-d", "/content/"], check=True)

CLIP = [l.strip() for l in (Path(DAVIS_ROOT) / "ImageSets/2017/val.txt").read_text().splitlines() if l.strip()][0]
ALL_FRAMES = sorted((Path(DAVIS_ROOT) / "JPEGImages/480p" / CLIP).glob("*.jpg"))

# Exp 2 sampling: 16 evenly spaced frames
EXP2_PATHS = [ALL_FRAMES[i] for i in np.linspace(0, len(ALL_FRAMES) - 1, 16, dtype=int)]
# Exp 3 sampling: every 12th frame
EXP3_PATHS = [ALL_FRAMES[i] for i in range(0, len(ALL_FRAMES), 12)]

def load_full(paths):
    """Original-resolution frames → (T, 3, H, W) float [0, 1]."""
    return torch.stack([T.ToTensor()(Image.open(p).convert("RGB")) for p in paths])

def load_pil_resized(paths, size=IMG_SIZE):
    """Frames resized with PIL exactly like Exp 2/3 → (T, 3, size, size) float [0, 1]."""
    tf = T.Compose([T.Resize((size, size)), T.ToTensor()])
    return torch.stack([tf(Image.open(p).convert("RGB")) for p in paths])

print(f"clip={CLIP}  total frames={len(ALL_FRAMES)}  exp2 T={len(EXP2_PATHS)}  exp3 T={len(EXP3_PATHS)}")

## 1 · Shapes, meta, guide alignment and normalization

In [ ]:
# ── Cell 4 · Generic adapter checks ─────────────────────────────────────────
def check_adapter(label, bb, frames, exp_t_out, exp_grid, exp_index, guide_size):
    out = bb(frames)
    f, g, m = out.features, out.guide, out.meta
    ok_shape = (f.shape[0] == exp_t_out and tuple(f.shape[-2:]) == exp_grid and f.shape[1] == m["feat_dim"]
                and tuple(g.shape) == (exp_t_out, 3, *guide_size))
    ok_index = m["frame_index"] == exp_index and m["t_out"] == exp_t_out and m["t_in"] == frames.shape[0]
    exp_guide = normalize_frames(resize_frames(frames[exp_index].to(DEVICE), guide_size), IMAGENET_MEAN, IMAGENET_STD)
    ok_guide = torch.allclose(g, exp_guide, atol=1e-6)
    ok_finite = bool(torch.isfinite(f).all())
    record(f"{label} shapes/meta/guide", ok_shape and ok_index and ok_guide and ok_finite,
           f"features={tuple(f.shape)} guide={tuple(g.shape)} frame_index={m['frame_index']} "
           f"shape={ok_shape} index={ok_index} guide_norm={ok_guide} finite={ok_finite}")
    print("   meta:", {k: v for k, v in m.items() if k != 'frame_index'})
    return out

full16 = load_full(EXP2_PATHS)   # (16, 3, 480, 854)
full5  = load_full(EXP3_PATHS[:5])

bb = BackboneLoader.load("videomae", device=DEVICE)
check_adapter("videomae T=16", bb, full16, 8, (14, 14), list(range(0, 16, 2)), (224, 224))
check_adapter("videomae guide_size=56", BackboneLoader.load("videomae", device=DEVICE, guide_size=56),
              full16, 8, (14, 14), list(range(0, 16, 2)), (56, 56))
try:
    bb(full5); record("videomae rejects T!=16", False, "no error raised")
except ValueError as e:
    record("videomae rejects T!=16", True, str(e))
del bb; torch.cuda.empty_cache()

bb = BackboneLoader.load("swin_t", device=DEVICE)
check_adapter("swin_t per_frame T=5", bb, full5, 5, (7, 7), list(range(5)), (224, 224))
del bb
bb = BackboneLoader.load("swin_t", device=DEVICE, temporal_mode="clip")
check_adapter("swin_t clip T=5 (odd)", bb, full5, 3, (7, 7), [0, 2, 4], (224, 224))
del bb; torch.cuda.empty_cache()

bb = BackboneLoader.load("dinov2", device=DEVICE)
check_adapter("dinov2 T=1 (image)", bb, full5[:1], 1, (16, 16), [0], (224, 224))
check_adapter("dinov2 T=5", bb, full5, 5, (16, 16), list(range(5)), (224, 224))
del bb; torch.cuda.empty_cache()

## 2 · VideoMAE vs Exp 2 code

In [ ]:
# ── Cell 5 · Exp 2 extraction, copied from results/exp2_temporal_consistency.ipynb ──
from transformers import VideoMAEModel
videomae = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base").to(DEVICE).eval()
preprocess = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
def extract_videomae_features(frame_paths):
    frames = torch.stack([preprocess(Image.open(p).convert("RGB")) for p in frame_paths], dim=0)
    inp = frames.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = videomae(pixel_values=inp)
    tokens = out.last_hidden_state.squeeze(0).cpu()
    feats  = tokens.reshape(8, 14, 14, 768).permute(0, 3, 1, 2).contiguous()
    guides = frames[::2]
    return feats, guides

old_feats, old_guides = extract_videomae_features(EXP2_PATHS)
del videomae; torch.cuda.empty_cache()

bb = BackboneLoader.load("videomae", device=DEVICE)
# (a) exact: same PIL-resized input
new = bb(load_pil_resized(EXP2_PATHS))
mx, cs = compare(new.features, old_feats)
record("videomae vs Exp2 (a) exact, PIL-resized input", mx < EXACT_MAX_ABS and cs > EXACT_MIN_COS, f"max_abs={mx:.2e} min_cos={cs:.6f}")
mx, cs = compare(new.guide, old_guides)
record("videomae guide vs Exp2 guides", mx < 1e-5, f"max_abs={mx:.2e}")
# (b) full path: original-resolution input, our tensor resize
new = bb(load_full(EXP2_PATHS))
mx, cs = compare(new.features, old_feats)
record("videomae vs Exp2 (b) full path (tensor vs PIL resize)", None, f"max_abs={mx:.2e} min_cos={cs:.6f}")
del bb, new; torch.cuda.empty_cache()

## 3 · Swin-T per-frame vs Exp 3 code

In [ ]:
# ── Cell 6 · Exp 3 extraction, copied from results/exp3_product_in_action_FIXED.ipynb ──
from torchvision.models.video import swin3d_t, Swin3D_T_Weights
_feat = {}
def _hook_fn(module, inp, out):
    _feat["norm"] = out.detach().permute(0, 4, 1, 2, 3).contiguous()
backbone = swin3d_t(weights=Swin3D_T_Weights.KINETICS400_V1).to(DEVICE).eval()
backbone.norm.register_forward_hook(_hook_fn)
frame_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
@torch.no_grad()
def extract_swin_features(sampled_frame_paths):
    imgs = [Image.open(p).convert("RGB") for p in sampled_frame_paths]
    x = torch.stack([frame_transform(im) for im in imgs])
    x = x.unsqueeze(2).repeat(1, 1, 2, 1, 1).to(DEVICE)
    _ = backbone(x)
    out = _feat["norm"]
    out = out.squeeze(2).contiguous().cpu()
    return out.float()

old_feats = extract_swin_features(EXP3_PATHS)
del backbone; torch.cuda.empty_cache()

bb = BackboneLoader.load("swin_t", device=DEVICE, norm="imagenet")   # Exp 3 used ImageNet stats
new = bb(load_pil_resized(EXP3_PATHS))
mx, cs = compare(new.features, old_feats)
record("swin_t per_frame vs Exp3 (a) exact, norm=imagenet", mx < EXACT_MAX_ABS and cs > EXACT_MIN_COS, f"max_abs={mx:.2e} min_cos={cs:.6f}")
new = bb(load_full(EXP3_PATHS))
mx, cs = compare(new.features, old_feats)
record("swin_t per_frame vs Exp3 (b) full path", None, f"max_abs={mx:.2e} min_cos={cs:.6f}")
del bb

bb = BackboneLoader.load("swin_t", device=DEVICE)   # default norm="kinetics"
new = bb(load_pil_resized(EXP3_PATHS))
mx, cs = compare(new.features, old_feats)
record("swin_t kinetics vs imagenet norm (expected to differ)", None, f"max_abs={mx:.2e} min_cos={cs:.6f}")
del bb, new; torch.cuda.empty_cache()

## 4 · DINOv2 (timm wrapper)

In [ ]:
# ── Cell 7 · Pretrained weights + sanity ────────────────────────────────────
import timm
bb = BackboneLoader.load("dinov2", device=DEVICE)
name = bb.timm_name
sd_bb = {k: v.cpu() for k, v in bb.model.model.state_dict().items()}

ref_pre  = timm.create_model(name, pretrained=True,  num_classes=0, dynamic_img_size=True).state_dict()
ref_rand = timm.create_model(name, pretrained=False, num_classes=0, dynamic_img_size=True).state_dict()
same_pre  = all(torch.equal(sd_bb[k], ref_pre[k]) for k in ref_pre)
diff_rand = any(not torch.equal(sd_bb[k], ref_rand[k]) for k in ref_rand if ref_rand[k].numel() > 1)
record("dinov2 weights == timm pretrained=True", same_pre, f"{len(ref_pre)} tensors compared ({name})")
record("dinov2 weights != timm pretrained=False", diff_rand, "random-init weights differ")
del ref_pre, ref_rand

img   = load_pil_resized(EXP3_PATHS[:1])           # (1, 3, 224, 224)
flip  = torch.flip(img, dims=[-1])
noise = torch.rand_like(img)
f_img   = bb(img).features.mean(dim=(-2, -1))      # (1, C) mean-pooled
f_flip  = bb(flip).features.mean(dim=(-2, -1))
f_noise = bb(noise).features.mean(dim=(-2, -1))
c_flip  = F.cosine_similarity(f_img, f_flip).item()
c_noise = F.cosine_similarity(f_img, f_noise).item()
record("dinov2 sanity: cos(img, flip) > cos(img, noise)", c_flip > c_noise, f"cos_flip={c_flip:.4f} cos_noise={c_noise:.4f}")
del bb; torch.cuda.empty_cache()

## Summary — paste this output back

In [ ]:
# ── Cell 8 · Summary ────────────────────────────────────────────────────────
import platform, transformers
print(f"torch={torch.__version__} torchvision={__import__('torchvision').__version__} "
      f"transformers={transformers.__version__} timm={timm.__version__} "
      f"gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
print(f"clip={CLIP}")
for name, passed, detail in RESULTS:
    tag = "INFO" if passed is None else ("PASS" if passed else "FAIL")
    print(f"[{tag}] {name}: {detail}")
n_fail = sum(p is False for _, p, _ in RESULTS)
print(f"\n{len(RESULTS)} checks, {n_fail} failed")